# Data Ingest + Scrape Notebook

Goal: investigate existing data, optionally scrape Ramayana verses from rahular.com, normalize to project schema, and write a consolidated JSONL that the GUI (`gui_app.py`) will load.

In [36]:
# Dependencies and setup
from pathlib import Path
import json, re, sys, shutil, datetime

ROOT = Path(r"c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp")
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEED_FILES = [
    PROCESSED_DIR / "bk1_sarga1_verses.jsonl",
    PROCESSED_DIR / "gui_verses.jsonl",
]

def read_jsonl(path: Path):
    """Read JSONL; return list of dicts. Ignores malformed lines."""
    if not path.exists():
        return []
    out = []
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    out.append(obj)
            except Exception:
                continue
    return out

def write_jsonl(path: Path, records):
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def parse_id_parts(vid: str):
    """Parse id like '1.1.3' -> (book, sarga, verse). Return strings."""
    if not vid or "." not in vid:
        return ("1", "1", "1")
    parts = vid.split(".")
    if len(parts) != 3:
        return (parts[0] if parts else "1", parts[1] if len(parts) > 1 else "1", parts[2] if len(parts) > 2 else "1")
    return parts[0], parts[1], parts[2]

def normalize_record(rec: dict):
    """Ensure minimal schema for GUI: id, sa_verse, en, book, sarga."""
    out = dict(rec)
    vid = str(out.get("id") or "").strip()
    if not vid:
        # try to infer an id if possible
        # fallback dummy id uses current timestamp
        ts = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
        vid = f"1.1.{ts}"
        out["id"] = vid
    book, sarga, _ = parse_id_parts(vid)
    out["book"] = out.get("book", book)
    out["sarga"] = out.get("sarga", sarga)
    out["sa_verse"] = out.get("sa_verse", "").strip()
    out["en"] = out.get("en", "").strip()
    return out

def merge_by_id(*record_lists, prefer="existing"):
    """
    Merge lists of records by 'id'.
    prefer = 'existing'|'newer'
    - 'existing': first seen wins
    - 'newer': later lists overwrite earlier for same id
    """
    merged = {}
    order = record_lists if prefer == "newer" else record_lists[::-1]
    for lst in order:
        for r in lst:
            rid = str(r.get("id") or "").strip()
            if not rid:
                continue
            merged[rid] = normalize_record(r)
    # restore final order by sorted id
    return [merged[k] for k in sorted(merged.keys(), key=lambda x: tuple(int(p) if p.isdigit() else 0 for p in x.split(".")))]

print("Setup complete.")

Setup complete.


In [37]:
# Investigate current processed files
report = {}
for p in SEED_FILES:
    rows = read_jsonl(p)
    report[str(p.name)] = {
        "exists": p.exists(),
        "count": len(rows),
        "first_3_ids": [r.get("id") for r in rows[:3]],
    }

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "bk1_sarga1_verses.jsonl": {
    "exists": true,
    "count": 5,
    "first_3_ids": [
      "1.1.1",
      "1.1.2",
      "1.1.3"
    ]
  },
  "gui_verses.jsonl": {
    "exists": true,
    "count": 19182,
    "first_3_ids": [
      "1.1.1",
      "1.1.2",
      "1.1.3"
    ]
  }
}


In [38]:
# Optional: Scraper for rahular.com (requests + BeautifulSoup)
# If libs missing, auto-install.
try:
    import requests
    from bs4 import BeautifulSoup
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "beautifulsoup4"])
    import requests
    from bs4 import BeautifulSoup

ROMAN_MAP = {1: "i", 2: "ii", 3: "iii", 4: "iv", 5: "v", 6: "vi"}

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
})

def _make_soup(html: bytes):
    # Try lxml first (if available), then html.parser
    try:
        return BeautifulSoup(html, "lxml")
    except Exception:
        return BeautifulSoup(html, "html.parser")

# Heuristics to classify text blocks
IAST_CHARS = set("āīūṛṝḷḹṃṁṅñṭḍṇśṣḥĀĪŪṚṜḶḸṀṄÑṬḌṆŚṢḤ")
BOILERPLATE = [
    "copyright", "all rights", "share", "posted", "comments",
    "next", "previous", "related", "recent posts", "subscribe",
]

DANDA = "\u0964"  # ।
DOUBLE_DANDA = "\u0965"  # ॥


def is_devanagari(text: str) -> bool:
    return any('\u0900' <= ch <= '\u097f' for ch in text)

def is_iast(text: str) -> bool:
    return any(ch in IAST_CHARS for ch in text)

def likely_english(text: str) -> bool:
    # Mostly ASCII letters, spaces, and punctuation; avoid dev/iast
    if is_devanagari(text) or is_iast(text):
        return False
    ascii_ratio = sum(1 for ch in text if ord(ch) < 128) / max(1, len(text))
    return ascii_ratio > 0.8

def clean_text(x) -> str:
    if hasattr(x, 'get_text'):
        s = x.get_text(" ", strip=True)
    else:
        s = str(x).strip()
    # normalize whitespace
    s = re.sub(r"\s+", " ", s)
    return s

def extract_blocks(soup: BeautifulSoup):
    # Prefer main content areas but fall back gracefully
    containers = soup.select(
        ", ".join([
            "div.post-body", "div.entry-content", "article", "div#content", "main",
            "div#main", "div.post", "div.page", "section", 
            "div.blog-posts", "div.blog-post", "div.post-outer", "div.post-content",
            "div.post-body.entry-content"
        ])
    )
    if not containers and soup.body:
        containers = [soup.body]
    if not containers:
        containers = [soup]

    blocks = []
    for c in containers:
        for el in c.select("p, li, div, pre, span, blockquote"):
            txt = clean_text(el)
            if not txt or len(txt) < 4:
                continue
            low = txt.lower()
            if any(b in low for b in BOILERPLATE):
                continue
            blocks.append(txt)
    # Fallback to global paragraphs if nothing found
    if not blocks:
        for el in soup.find_all(["p", "li", "div", "pre", "span", "blockquote"]):
            txt = clean_text(el)
            if not txt or len(txt) < 4:
                continue
            low = txt.lower()
            if any(b in low for b in BOILERPLATE):
                continue
            blocks.append(txt)
    return blocks


KANDA_HEADING_RE = re.compile(r"\b(Bala|Ayodhya|Araṇya|Aranya|Kishkindha|Sundara|Yuddha)\s+Kanda\s*:\s*Chapter\s*\d+\b", re.I)


def split_bilingual_pairs(text: str):
    """
    Extract Sanskrit–English pairs from a single combined text using danda delimiters
    and Devanagari→English boundary detection. Handles cases like:
    '...॥English sentence. ...॥Next English...'
    Returns list of {sanskrit, english}.
    """
    pairs = []
    if not text:
        return pairs
    # Remove explicit Kanda headings to reduce noise
    t = KANDA_HEADING_RE.sub(" ", text)
    # Normalize some spacing
    t = t.replace(" | ", " ")

    i = 0
    n = len(t)
    while i < n:
        # find start of next Devanagari chunk
        j = i
        while j < n and not ('\u0900' <= t[j] <= '\u097f'):
            j += 1
        if j >= n:
            break
        # find end of Sanskrit chunk: advance until after last danda(s)
        k = j
        last_dev = j
        while k < n and ('\u0900' <= t[k] <= '\u097f' or t[k] in (DANDA, DOUBLE_DANDA) or t[k].isdigit() or t[k].isspace() or t[k] in ",.;:()[]'\"-–—/\\“”’‘"):
            if '\u0900' <= t[k] <= '\u097f':
                last_dev = k
            k += 1
        sa = t[j:k].strip()
        # If we have explicit danda marks, cut Sanskrit up to them
        dd_pos = max(sa.rfind(DOUBLE_DANDA), sa.rfind(DANDA))
        if dd_pos != -1:
            sa = sa[:dd_pos+1]
        sa = sa.strip()
        # Now collect English from k until next Devanagari start
        m = k
        while m < n and not ('\u0900' <= t[m] <= '\u097f'):
            m += 1
        en = t[k:m].strip()
        en = re.sub(r"^[\s\-–—:·•]+", "", en)
        # Clean and validate
        if sa and is_devanagari(sa) and en and not is_devanagari(en):
            if not KANDA_HEADING_RE.search(en):
                pairs.append({"sanskrit": sa, "english": en})
        i = m if m > k else k + 1
    return pairs


def scrape_ramayana_sarga(book_num: int, sarga_num: int):
    """
    Scrapes a specific sarga from rahular.com and returns a list of
    {"sanskrit": "...", "english": "..."} pairs.

    Strategy:
    - Extract content blocks and merge to one text string.
    - First try bilingual line splitting using danda (॥/।) and boundary detection.
    - If none found, fall back to block-based Sanskrit→English accumulation.
    - If still empty, fetch the mobile version (?m=1) and retry both strategies.
    """
    book_roman = ROMAN_MAP.get(book_num)
    if not book_roman:
        print(f"[warn] book_num {book_num} out of range (1-6)")
        return []

    def fetch_and_parse(url: str):
        try:
            resp = SESSION.get(url, timeout=30)
            if resp.status_code == 404:
                print("[info] 404 not found")
                return []
            resp.raise_for_status()
            html = resp.content
        except Exception as e:
            print(f"[error] fetch failed: {e}")
            return []

        soup = _make_soup(html)
        blocks = extract_blocks(soup)
        if not blocks:
            return []

        combined = "\n".join(blocks)
        pairs = split_bilingual_pairs(combined)
        if pairs:
            print(f"[parse-inline] pairs: {len(pairs)}")
            return pairs

        # Fallback: block-wise accumulation
        pairs = []
        sa_current = None
        en_buf = []

        def commit():
            nonlocal sa_current, en_buf
            if sa_current:
                en_text = " ".join(en_buf).strip()
                # Allow diacritics in English; only exclude Devanagari
                if en_text and not is_devanagari(en_text):
                    if not KANDA_HEADING_RE.search(en_text):
                        pairs.append({"sanskrit": sa_current.strip(), "english": en_text})
            sa_current, en_buf = None, []

        for blk in blocks:
            if is_devanagari(blk) or is_iast(blk):
                commit()
                sa_current = blk
                en_buf = []
            else:
                if sa_current:
                    en_buf.append(blk)

        commit()
        if pairs:
            print(f"[parse-fallback] pairs: {len(pairs)}")
        return pairs

    base_url = f"http://www.rahular.com/itihasa/gen/ramayana/vol-{book_roman}/{sarga_num}.html"
    print(f"[fetch] {base_url}")
    pairs = fetch_and_parse(base_url)
    if pairs:
        return pairs

    mobile_url = base_url + "?m=1"
    print(f"[fetch-mobile] {mobile_url}")
    pairs = fetch_and_parse(mobile_url)
    return pairs


def to_project_records(pairs, book_num: int, sarga_num: int):
    """Map scraped pairs to project schema (id, sa_verse, en, book, sarga)."""
    out = []
    for i, v in enumerate(pairs, start=1):
        rid = f"{book_num}.{sarga_num}.{i}"
        out.append({
            "id": rid,
            "book": str(book_num),
            "sarga": str(sarga_num),
            "sa_verse": v.get("sanskrit", "").strip(),
            "en": v.get("english", "").strip(),
            "source": "scrape_rahular",
            "version": "1.0",
        })
    return out

print("Scraper functions ready.")

Scraper functions ready.


In [39]:
# Bulk scrape across books and sargas; fall back to sample if nothing scraped
import time

DO_SCRAPE = True
USE_SAMPLE_DATA = False
MAX_EMPTY_STREAK = 5
RETRIES = 2
PER_SARGA_DELAY_SEC = 0.2  # polite delay


def scrape_book_sarga_range(book_start=1, book_end=6, sarga_max_guess=220):
    all_records = []
    for book in range(book_start, book_end + 1):
        empty_streak = 0
        for sarga in range(1, sarga_max_guess + 1):
            pairs = []
            for attempt in range(1, RETRIES + 2):
                pairs = scrape_ramayana_sarga(book, sarga)
                if pairs:
                    break
                if attempt <= RETRIES:
                    print(f"[retry] book {book} sarga {sarga} attempt {attempt}/{RETRIES}")
            if not pairs:
                empty_streak += 1
                if empty_streak >= MAX_EMPTY_STREAK:
                    print(f"[info] stopping book {book} after {sarga-1} sargas (empty streak)")
                    break
                time.sleep(PER_SARGA_DELAY_SEC)
                continue
            empty_streak = 0
            records = to_project_records(pairs, book, sarga)
            print(f"[ok] book {book} sarga {sarga}: {len(records)} verses")
            all_records.extend(records)

            # incremental write to avoid data loss
            TEMP_TARGET = PROCESSED_DIR / f"_tmp_gui_verses_{book}_{sarga}.jsonl"
            write_jsonl(TEMP_TARGET, all_records)
            time.sleep(PER_SARGA_DELAY_SEC)
    return all_records

scraped_preview = []
if DO_SCRAPE:
    # Scrape all 6 volumes; high upper bound per volume, stops via empty-streak
    scraped_preview = scrape_book_sarga_range(book_start=1, book_end=6, sarga_max_guess=220)
    if not scraped_preview and not USE_SAMPLE_DATA:
        print("[warn] scraping produced no data; consider enabling USE_SAMPLE_DATA")
elif USE_SAMPLE_DATA:
    print("[info] using sample Sanskrit data")
    scraped_preview = to_project_records(SAMPLE_VERSES, 1, 1)

print("[data] sample:")
print(json.dumps(scraped_preview[:3], indent=2, ensure_ascii=False))

[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/1.html
[parse-inline] pairs: 79
[ok] book 1 sarga 1: 79 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/2.html
[parse-inline] pairs: 39
[ok] book 1 sarga 2: 39 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/2.html
[parse-inline] pairs: 39
[ok] book 1 sarga 2: 39 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/3.html
[parse-inline] pairs: 30
[ok] book 1 sarga 3: 30 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/3.html
[parse-inline] pairs: 30
[ok] book 1 sarga 3: 30 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/4.html
[parse-inline] pairs: 20
[ok] book 1 sarga 4: 20 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/4.html
[parse-inline] pairs: 20
[ok] book 1 sarga 4: 20 verses
[fetch] http://www.rahular.com/itihasa/gen/ramayana/vol-i/5.html
[parse-inline] pairs: 19
[ok] book 1 sarga 5: 19 verses
[fetch] http://www.rahular.com/i

In [40]:
# Merge: seed data + GUI saves + (optional) scraped
seed_records = []
for p in SEED_FILES:
    seed_records.extend(read_jsonl(p))

merged = merge_by_id(seed_records, scraped_preview, prefer="newer")
print(f"[merge] total merged records: {len(merged)}")
print("[merge] sample:")
print(json.dumps(merged[:2], indent=2, ensure_ascii=False))

[merge] total merged records: 19182
[merge] sample:
[
  {
    "id": "1.1.1",
    "book": "1",
    "sarga": "1",
    "sa_verse": "ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वरम्। नारदं परिपप्रच्छ वाल्मीकिर्मुनिपुङ्गवम्॥",
    "en": "The ascetic Vālmīki asked Nārada, the best of sages and foremost of those conversant with words, ever engaged in austerities and Vedic studies.",
    "source": "scrape_rahular",
    "version": "1.0"
  },
  {
    "id": "1.1.2",
    "book": "1",
    "sarga": "1",
    "sa_verse": "कोन्वस्मिन् साम्प्रतं लोके गुणवान् कश्च वीर्यवान्। धर्मज्ञश्च कृतज्ञश्च सत्यवाक्यो दृढत्नतः॥",
    "en": "Who at present in this world is like crowned with qualities, and with prowess, knowing duty, and grateful, and truthful, and firm in vow.",
    "source": "scrape_rahular",
    "version": "1.0"
  }
]


In [41]:
# Write consolidated dataset with backup-and-replace option
TARGET = PROCESSED_DIR / "gui_verses.jsonl"
BACKUP_DIR = PROCESSED_DIR / "backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def backup_file(path: Path):
    if path.exists():
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        bak = BACKUP_DIR / f"{path.stem}_{ts}{path.suffix}"
        shutil.copy2(path, bak)
        print(f"[backup] saved {bak}")

# Save consolidated data
backup_file(TARGET)
write_jsonl(TARGET, merged)
print(f"[write] wrote {len(merged)} records to {TARGET}")

[backup] saved c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\backups\gui_verses_20250817_211355.jsonl
[write] wrote 19182 records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl
[write] wrote 19182 records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl


In [34]:
# Ingest local structured file via existing parser and merge into GUI dataset
from importlib import import_module

try:
    parse_mod = import_module('scripts.parse_ramayana_txt')
except Exception as e:
    # Add ROOT/src to path if needed
    import sys
    sys.path.append(str(ROOT))
    parse_mod = import_module('scripts.parse_ramayana_txt')

input_txt = DATA_DIR / 'ramayana_translation_data.txt'
if input_txt.exists():
    print(f"[local-parse] parsing {input_txt}")
    verses = parse_mod.parse_ramayana_file(str(input_txt))
    print(f"[local-parse] got {len(verses)} verses")
    parsed_path = PROCESSED_DIR / 'parsed_local_ramayana.jsonl'
    write_jsonl(parsed_path, verses)
    print(f"[local-parse] wrote {parsed_path}")

    # Map to GUI-minimal schema and merge
    parsed_gui = []
    for v in verses:
        rid = str(v.get('id') or '').strip() or '1.1.0'
        book = str(v.get('book') or '1')
        sarga = str(v.get('sarga') or '1')
        parsed_gui.append({
            'id': rid,
            'book': book,
            'sarga': sarga,
            'sa_verse': v.get('sa_verse', ''),
            'en': v.get('en', ''),
            'source': 'local_parsed',
            'version': '1.0'
        })

    merged2 = merge_by_id(merged, parsed_gui, prefer='newer')
    backup_file(TARGET)
    write_jsonl(TARGET, merged2)
    print(f"[write] merged {len(merged2)} total records to {TARGET}")
else:
    print(f"[skip] {input_txt} not found; local parse skipped")

[local-parse] parsing c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\ramayana_translation_data.txt
[local-parse] got 5 verses
[local-parse] wrote c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\parsed_local_ramayana.jsonl
[backup] saved c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\backups\gui_verses_20250817_210354.jsonl
[write] merged 19182 total records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl
[write] merged 19182 total records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl


## Usage Notes

- After running Cell 7, launch the GUI:
  - powershell: `streamlit run gui_app.py`
- The app loads `data/processed/gui_verses.jsonl` automatically if present; your merged dataset will be available in the sidebar "Saved Verses."
- To scrape more content:
  - Set `DO_SCRAPE=True` in Cell 5 and adjust `SCRAPE_BOOK`, `SCRAPE_SARGA`.
  - Re-run Cells 5–7 to merge and save.

## Itihasa dataset ingestion (compliant source)

- This project can ingest the Itihasa Sanskrit–English corpus (Aralikatte et al., 2021) as a primary, properly cited source.
- Please ensure SOURCES.md includes the Itihasa citation (added by this update) to stay compliant.
- Usage options:
  - Place downloaded Itihasa Ramayana JSON/JSONL files under `data/external/itihasa/` and run the next cell.
  - Optionally set `ITIHASA_URL` to a direct ZIP or JSONL URL for automatic download (will be saved under `data/external/itihasa/`).
- The loader normalizes records into the GUI schema: `id=book.sarga.verse`, `sa_verse`, `en`, `book`, `sarga`, and merges into `data/processed/gui_verses.jsonl` with backup.
- Fields expected (flexible): `sanskrit|sa|sloka`, `english|en`, and preferably `book`, `sarga`, `verse` or a composite `id`. The code attempts to infer missing parts.

In [35]:
# Ingest Itihasa dataset (Sanskrit–English pairs) with proper citation recorded
# Source: Aralikatte et al., 2021 (WAT2021) — see SOURCES.md cell below.
from pathlib import Path
import json, zipfile

EXTERNAL_DIR = DATA_DIR / 'external' / 'itihasa'
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set a direct URL to download a ZIP/JSONL from Itihasa mirrors
ITIHASA_URL = ''  # e.g., 'https://example.com/itihasa_ramayana.jsonl.zip'
DOWNLOAD_PATH = EXTERNAL_DIR / 'itihasa_download.tmp'

try:
    import requests
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests'])
    import requests

if ITIHASA_URL:
    print(f"[download] {ITIHASA_URL}")
    try:
        with requests.get(ITIHASA_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(DOWNLOAD_PATH, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        # if ZIP, extract
        if str(DOWNLOAD_PATH).lower().endswith('.zip'):
            with zipfile.ZipFile(DOWNLOAD_PATH, 'r') as zf:
                zf.extractall(EXTERNAL_DIR)
            print(f"[download] extracted to {EXTERNAL_DIR}")
        else:
            print(f"[download] saved to {DOWNLOAD_PATH}")
    except Exception as e:
        print(f"[warn] download failed: {e}")

# Discover candidate files (json/jsonl) under EXTERNAL_DIR
candidates = list(EXTERNAL_DIR.glob('**/*.json')) + list(EXTERNAL_DIR.glob('**/*.jsonl'))
print(f"[scan] found {len(candidates)} candidate files under {EXTERNAL_DIR}")

# Heuristic: keep only Ramayana files if path hints exist; else keep all
ramayana_candidates = [p for p in candidates if any(k in p.name.lower() for k in ['ramayana', 'ramayan'])] or candidates
print(f"[scan] using {len(ramayana_candidates)} files for ingestion")

# Helper: safe load of json or jsonl

def load_any(path: Path):
    data = []
    try:
        if path.suffix.lower() == '.jsonl':
            with path.open('r', encoding='utf-8', errors='ignore') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        data.append(json.loads(line))
                    except Exception:
                        continue
        else:
            data = json.loads(path.read_text(encoding='utf-8', errors='ignore'))
            if isinstance(data, dict):
                # some dumps wrap in a top-level object
                # try common keys
                for k in ['data', 'records', 'items']:
                    if k in data and isinstance(data[k], list):
                        data = data[k]
                        break
                if isinstance(data, dict):
                    data = [data]
        return data
    except Exception as e:
        print(f"[warn] failed to load {path}: {e}")
        return []

# Field extraction helpers
SA_KEYS = ['sanskrit', 'sa', 'sloka', 'shloka', 'śloka', 'verse_sa']
EN_KEYS = ['english', 'en', 'translation', 'verse_en']


def first(d, keys, default=''):
    for k in keys:
        if k in d and d[k]:
            return d[k]
    return default


def coerce_int_str(x, default='1'):
    try:
        if x is None:
            return default
        s = str(x).strip()
        if not s:
            return default
        # extract leading integer
        m = re.match(r'\d+', s)
        return m.group(0) if m else default
    except Exception:
        return default

# Build normalized records
itihasa_records = []
for fp in ramayana_candidates:
    rows = load_any(fp)
    print(f"[load] {fp.name}: {len(rows)} rows")
    for r in rows:
        sa = first(r, SA_KEYS, '')
        en = first(r, EN_KEYS, '')

        # Infer id from explicit fields or composite
        book = coerce_int_str(r.get('book', r.get('kanda')))
        sarga = coerce_int_str(r.get('sarga', r.get('chapter')))
        verse = coerce_int_str(r.get('verse', r.get('sloka_id')))
        rid = str(r.get('id') or '').strip()
        if not rid:
            rid = f"{book}.{sarga}.{verse}"
        nb, ns, _ = parse_id_parts(rid)

        itihasa_records.append({
            'id': rid,
            'book': nb,
            'sarga': ns,
            'sa_verse': (sa or '').strip(),
            'en': (en or '').strip(),
            'source': 'itihasa',
            'version': '1.0'
        })

print(f"[normalize] built {len(itihasa_records)} normalized records from Itihasa")

# Merge into existing merged data and write
merged3 = merge_by_id(merged, itihasa_records, prefer='newer')
backup_file(TARGET)
write_jsonl(TARGET, merged3)
print(f"[write] merged {len(merged3)} total records to {TARGET}")

[scan] found 0 candidate files under c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\external\itihasa
[scan] using 0 files for ingestion
[normalize] built 0 normalized records from Itihasa
[backup] saved c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\backups\gui_verses_20250817_210354.jsonl
[write] merged 19182 total records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl
[write] merged 19182 total records to c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\gui_verses.jsonl
